# 02 · Data cleaning and validation
Re-runs the cleaning functions on the raw data and inspects the exclusion log, identity checks and model-artifact screen. The pipeline itself is `python scripts/prepare_data.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30, "display.width", 160)

from src.data import load
from src.analysis import descriptive, trends, seasonality, correlations, statistical_tests as st, data_quality as dq, sensitivity
from src.visualization import charts, maps
from src.data.transform import independent_units

wb = descriptive.real_only(load.load_processed("wb_daily"))
cities = load.load_processed("wb_cities")

## Exclusion and transformation log

In [2]:
load.load_exclusions()

,source_id,step,rule,column,n_rows,action,example
0,wb_pm25,identity_check,place name shared/near-identical across differ...,city,8,flagged (kept); labels disambiguated,"Sahiwal (31.97°N, 72.33°E), Sahiwal (30.67°N, ..."
1,wb_pm25,summary,rows retained after cleaning,NaN,948183,"retained 948,183 of 948,183",NaN
2,wb_pm25,outlier_flag,robust |z| > 3.5 on log(PM2.5) within city,pm25,783,flagged (kept),NaN
3,wb_pm25,identity_check,identical daily PM2.5 series shared by several...,pm25,29,flagged (kept); inferential tests use one repr...,"Zaida, Tordher, Swabi, Sukkur, Sinjhoro, Sarai..."
4,wb_pm25,model_artifact_screen,abrupt level shift,pm25,101,city flagged (kept); optional exclusion filter,"Chuhar Jamali, Pir Jo Goth, Khairpur Mir's, Za..."
5,wb_pm25,model_artifact_screen,fixed annual mean across years,pm25,6,city flagged (kept); optional exclusion filter,"Sialkot, Mustafabad, Hadali, Dipalpur, Chunian"
6,wb_pm25,model_artifact_screen,fixed annual mean + abrupt level shift,pm25,5,city flagged (kept); optional exclusion filter,"Sukkur, Rohri, Rahim Yar Khan, Multan, Mirpur ..."
7,hf_aq,transformation,source-derived calendar/AQI columns dropped; c...,"date,hour,day_of_week,month,month_name,year,is...",21840,columns removed,NaN
8,hf_aq,summary,rows retained after cleaning,NaN,21840,"retained 21,840 of 21,840",NaN
9,hf_aq,aggregation,hourly->daily requires >= 18 valid hours,NaN,0,no days affected,NaN


## Schema validation of processed tables

In [3]:
from src.data.validate import validate_schema
for name in ['wb_daily','wb_cities','hf_hourly','hf_daily','weather_daily']:
    r = validate_schema(load.load_processed(name), name)
    print(f'{name:15s}', 'OK' if r.ok else r.issues)

wb_daily        OK
wb_cities       OK
hf_hourly       OK
hf_daily        OK
weather_daily   OK


## Missing dates
Which calendar dates are absent from the World Bank series?

In [4]:
gaps = dq.find_gaps(wb['date'])
print(dq.missing_dates_pattern(gaps))
gaps.head()

{'n_missing': 94, 'all_month_end': True, 'share_month_end': 1.0, 'max_run': 1}


,gap_start,gap_end,length
0,2018-10-31,2018-10-31,1
1,2018-11-30,2018-11-30,1
2,2018-12-31,2018-12-31,1
3,2019-01-31,2019-01-31,1
4,2019-02-28,2019-02-28,1


In [5]:
charts.gap_strip(gaps, wb['date'].min(), wb['date'].max())

## Place identity checks
Shared names, identical model series and province-code consistency.

In [6]:
cities[cities['name_ambiguous']][['city_label','province','latitude','longitude','population']]

,city_label,province,latitude,longitude,population
70,"Sahiwal (31.97°N, 72.33°E)",Punjab,31.97386,72.33109,37186
71,"Sahiwal (30.67°N, 73.1°E)",Punjab,30.66595,73.10186,538344
313,"Bahawalnagar (30.0°N, 73.25°E)",Punjab,29.99835,73.25272,241873
317,Arifwala,Punjab,30.29058,73.06574,87360
319,Alipur (Islamabad Capital Territory),Islamabad Capital Territory,33.64894,73.17787,17566
320,Alipur (Punjab),Punjab,29.38242,70.91106,33601
324,"Bahawalnagar (30.55°N, 73.39°E)",Punjab,30.55083,73.39083,126700
336,Arif Wala,Balochistan,26.32478,66.29587,157063


In [7]:
cities[cities['series_group_size'] > 1].sort_values('series_group')[['series_group','city_label','population','is_series_representative']]

,series_group,city_label,population,is_series_representative
335,2,Nowshera Kalan,88003,False
318,2,Amangarh,42123,False
325,2,Nowshera Cantonment,96766,True
5,8,Zaida,26787,False
38,8,Swabi,97363,True
222,23,Jahangira,57011,False
20,23,Tordher,150000,True
40,42,Sukkur,563851,True
73,42,Rohri,50649,False
45,47,Sinjhoro,15970,False


In [8]:
pc = dq.province_consistency(cities)
pc[~pc['consistent']]

,city_id,city_label,adm1_code,province,neighbour_majority_adm1,nearest_km,consistent
4,wb:1162105,Zhob,PAK002,Balochistan,PAK005,115.967027,False
15,wb:1162862,Uthal,PAK002,Balochistan,PAK007,56.046622,False
16,wb:1162868,Usta Muhammad,PAK002,Balochistan,PAK007,7.929861,False
64,wb:1166066,Sarai Naurang,PAK005,Khyber Pakhtunkhwa,PAK006,24.239579,False
72,wb:1166652,Saddiqabad,PAK006,Punjab,PAK007,21.246333,False
117,wb:1169684,Murree,PAK006,Punjab,PAK005,21.189296,False
131,wb:1170564,Mehrabpur,PAK002,Balochistan,PAK007,7.929861,False
157,wb:1172513,Lachi,PAK005,Khyber Pakhtunkhwa,PAK006,24.447932,False
173,wb:1173664,Khuzdar,PAK002,Balochistan,PAK007,123.605879,False
224,wb:1176615,Islamabad,PAK003,Islamabad Capital Territory,PAK006,13.811442,False


## Model-artifact screen
Two documented diagnostics: repeated annual means and abrupt level shifts.

In [9]:
art = cities[cities['artifact_flag']]
print(len(art), 'of', len(cities), 'places flagged')
art['artifact_reason'].value_counts()

112 of 339 places flagged


artifact_reason
abrupt level shift                        101
fixed annual mean across years              6
fixed annual mean + abrupt level shift      5
Name: count, dtype: int64

In [10]:
art.sort_values('fixed_annual_run', ascending=False)[['city_label','province','fixed_annual_run','fixed_annual_from','fixed_annual_to','break_date','break_shift_rel']].head(15)

,city_label,province,fixed_annual_run,fixed_annual_from,fixed_annual_to,break_date,break_shift_rel
284,Chunian,Punjab,7,2019.0,2025.0,2021-11-01,0.045371
73,Rohri,Sindh,6,2019.0,2024.0,2025-01-01,-0.342324
40,Sukkur,Sindh,6,2019.0,2024.0,2025-01-01,-0.342324
81,Rahim Yar Khan,Punjab,6,2019.0,2024.0,2025-09-01,-0.364459
119,Multan,Punjab,5,2019.0,2023.0,2024-01-01,-0.541876
240,Hadali,Punjab,4,2019.0,2022.0,2024-01-01,0.119933
48,Sialkot,Punjab,4,2019.0,2022.0,2024-01-01,0.115722
116,Mustafabad,Punjab,3,2023.0,2025.0,2021-11-01,0.066689
323,Abbottabad,Khyber Pakhtunkhwa,3,2019.0,2021.0,2024-01-01,0.120188
125,Mirpur Khas,Sindh,3,2019.0,2021.0,2022-01-01,-1.517540


In [11]:
charts.artifact_annual_chart(trends.annual_series(wb), cities[cities['fixed_annual_run'] >= 3])

In [12]:
charts.break_date_chart(cities)

## Outlier flags
Robust z-score on log PM2.5 within each city; flagged, never deleted.

In [13]:
dq.outlier_summary(wb).head(10)

,city_label,n_outliers,n,outlier_pct
37,Chaman,171,2797,6.113693
303,Taftan,109,2797,3.897033
184,Mach,74,2797,2.645692
250,Pishin,71,2797,2.538434
58,Dera Bugti,53,2797,1.894887
94,Hangu,44,2797,1.573114
136,Kamra,42,2797,1.501609
215,Murree,34,2797,1.215588
29,Buni,31,2797,1.108330
195,Mansehra,26,2797,0.929567


## Kaggle/HF timestamp diagnostic

In [14]:
hf = load.load_processed('hf_hourly')
al = dq.diurnal_peak_alignment(hf)
{k: v for k, v in al.items() if k != 'profile'}

{'pollutant': 'ozone',
 'weather': 'temperature',
 'pollutant_peak_hour': 9,
 'weather_peak_hour': 15,
 'pollutant_min_hour': 2,
 'weather_min_hour': 7,
 'offset_hours': 6}

In [15]:
charts.diurnal_alignment_chart(al['profile'])